# 演示：如何 predict

本 notebook 讲**打分是怎么算出来的**，逐层拆开 `competition_predict.ipynb` 里那一句
`main(DATASOURCES, start_date, end_date)`。用的是**同一个**
`competition_all_in_one.py`、同一份权重、同一套函数，所以口径与提交版完全一致。

| | `competition_predict.ipynb`（提交用） | 本 notebook（演示用） |
| --- | --- | --- |
| 代码来源 | `competition_all_in_one.py` | **同一个文件** |
| 打分入口 | `main(DATASOURCES, start, end)` | 同一个函数，外加逐步拆解 |
| 权重 | `MODEL_PATH` 的 JSON | 同一份 |
| 输出 | `['date','instrument','score']` | 同列同顺序 |

**推理侧只做一件事：加载权重打分，绝不训练。** 标准化统计随权重存盘、这里原样复用，
不用测试区间自己的数据重新估计。

**哪些 cell 需要平台数据**：加载权重、四道断言、锚点口径、`N` 覆盖这几步都是纯本地
计算，**离线可跑**；只有真正调 `main` 打分那一步需要 `dai.query`。下面逐个标注。

In [ ]:
# ---------- 导入自包含单文件 (与提交用 notebook 完全一致的那几行) ----------
import sys
from pathlib import Path

for _p in (Path.cwd(), Path.cwd() / 'notebooks', Path.cwd().parent):
    if (_p / 'competition_all_in_one.py').is_file():
        sys.path.insert(0, str(_p))
        break
else:
    raise FileNotFoundError('找不到 competition_all_in_one.py, 请与本 notebook 放在同一目录')

import numpy as np
import polars as pl
import torch
import competition_all_in_one as A

A.print_config_summary()

## 第 1 步：加载权重

权重是文本类 JSON（平台不接受 `.pt`）。`load_model` 按 `dtype`/`shape` 把每个张量
还原回来，非张量字段（结构超参、标准化统计、`anchor_slots`）原样读回。

In [ ]:
# [离线可跑] 加载提交用的那份权重, 看清里面有什么
ckpt = A.load_model(A.MODEL_PATH, map_location='cpu')

print('权重文件 :', A.MODEL_PATH.name)
print('张量数   :', len(ckpt['state_dict']),
      '| 参数量:', f"{sum(v.numel() for v in ckpt['state_dict'].values()):,}")
print()
print('推理必需的四类字段:')
print(f"  1. 结构超参 model_cfg = {ckpt['model_cfg']}")
print(f"  2. 窗口长度 seq_len   = {ckpt['seq_len']} (= lookback_days x 每天 bar 数)")
print(f"  3. 标准化   mean/std  = 各 {len(ckpt['mean'])} 维; "
      f"y_mean={ckpt['y_mean']:.6f} y_std={ckpt['y_std']:.6f}")
print(f"  4. 训练时刻 anchor_slots = {ckpt['anchor_slots']} (决定能在哪些 slot 上打分)")
print()
print(f"训练模式 : {ckpt['training_mode']} (finetune = 从预训练权重微调而来)")

## 第 2 步：四道断言 —— 配置与权重必须匹配

这是整个推理侧最重要的一环。配置和权重一旦错配，**分数照样算得出来，只是错的** ——
所以宁可直接报错，绝不静默出错。

| 断言 | 错了会怎样 |
| --- | --- |
| `feature_cols` 与内联 `dense_dataloader` 一致 | 顺序错 → 特征错位，输入喂错列 |
| `seq_len` 与配置算出的 `lookback_bars` 一致 | 长度错 → 窗口错位 |
| `PREDICT_SLOT` 在 **checkpoint 的** `anchor_slots` 里 | 在模型没训练过的时刻上外推 |
| `PREDICT_SLOT` 在 **配置的** `finetune.anchor_slots` 里 | 压根不会生成该 slot 的锚点 |

后两条是一对：前者管"模型见过这个时刻吗"，后者管"数据管线会产出这个时刻吗"。

In [ ]:
# [离线可跑] 四道断言全部通过 (真实权重 + 提交用配置)
cfg = A._load_online_config()          # 内联配置 + 切成 online 口径
bars_per_day = A._check_checkpoint_matches_config(ckpt, cfg, A.CONFIG_NAME)

print('四道断言通过')
print(f'  bars_per_day  = {bars_per_day}  (5m 采样, 一天 240/5 = 48 根 bar)')
print(f'  PREDICT_SLOT  = {A.PREDICT_SLOT}   (slot 47 = 15:00 收盘)')
print(f'  ckpt.anchor_slots   = {ckpt["anchor_slots"]}')
print(f'  cfg.finetune.anchor_slots = {cfg.finetune.anchor_slots}')
print(f'  seq_len = {ckpt["seq_len"]} = lookback_days({cfg.data.lookback_days}) x {bars_per_day}')

In [ ]:
# [离线可跑] 反面演示: 用**预训练**配置 (无 finetune 段) 会被第 4 道断言挡下来
wrong = A._load_online_config('rwkv_256_softspearman.yaml')
try:
    A._check_checkpoint_matches_config(ckpt, wrong, 'rwkv_256_softspearman.yaml')
    print('不应该走到这里')
except AssertionError as e:
    print('按预期报错, 未静默算出错误分数:')
    print(' ', str(e).split(chr(10))[0])

## 第 3 步：为什么每天必须固定用 slot 47

这是最容易出错、且**错了不报错**的地方，值得单独讲。

平台要求的输出粒度是"每天每只股票一个分数"，但内部锚点本身是**日内多个采样时刻**。
所以要从锚点里挑出每天代表当日的那一个。

选法有两种，差别很关键：

- ❌ **每天取最大 `t_idx`** —— 依赖"锚点集合恰好以 47 收尾"这个隐含前提。
  换成 `stride: 3` 的预训练配置时，锚点是 0, 3, …, 45，每天最后一个是
  **slot 45（14:50）**，于是静默打在 14:50 上，而模型是在 46/47 上微调的。
- ✅ **按 `slot_idx == PREDICT_SLOT` 精确过滤** —— 口径不对时直接返回空表或断言失败，
  不会算出"看起来正常但时点错了"的分数。

单文件里用的是后者。下面用合成 `time_grid` 把两种选法的差别直接跑出来。

In [ ]:
# [离线可跑] 两种选法的差别: 同一份 time_grid, 一个打在 47, 一个静默打在 45
n_days = 20
grid = pl.DataFrame({
    'day_idx': np.repeat(np.arange(n_days), bars_per_day),
    'slot_idx': np.tile(np.arange(bars_per_day), n_days),
}).with_columns(
    (pl.col('day_idx') * bars_per_day + pl.col('slot_idx')).alias('t_idx'),
    (pl.datetime(2024, 1, 2) + pl.duration(days=pl.col('day_idx'))).alias('date'),
)
lookback = cfg.data.lookback_days * bars_per_day

# (a) 提交版口径: anchor_slots=[46,47] + 按 slot_idx 精确取
good = A.build_anchors(grid, 0, n_days - 1, lookback, 1,
                       anchor_slots=cfg.finetune.anchor_slots,
                       stride_by_day=True, day_stride=cfg.data.stride // bars_per_day)
sel_day, sel_t = A._select_daily_anchors({'anchors': good, 'time_grid': grid}, 'x', 'y')
print(f'(a) 提交版: {len(sel_day)} 天, 每天 1 个锚点, '
      f'命中 slot = {sorted(set(int(t) % bars_per_day for t in sel_t))}  <- 正确')

# (b) 旧口径: 预训练配置 stride=3 + "每天取最大 t_idx"
old = A.build_anchors(grid, 0, n_days - 1, lookback, 3)
worst = (grid.filter(pl.col('t_idx').is_in(old.tolist()))
             .group_by('day_idx').agg(pl.col('slot_idx').max()).sort('day_idx'))
print(f'(b) 旧口径: 每天取最大 t_idx -> '
      f'命中 slot = {sorted(worst["slot_idx"].unique().to_list())}  <- 静默打在 14:50, 错了')

In [ ]:
# [离线可跑] 口径不对时返回空表, 而不是"挑一个凑数"
empty = {'anchors': np.array([], np.int64), 'time_grid': grid}
print('无可用锚点时 _select_daily_anchors ->', A._select_daily_anchors(empty, 'a', 'b'))
print('predict_scores 收到 (None, None) 就返回空的 [date, instrument, score] 表')

### 一个锚点对应的输入窗口

`PREDICT_SLOT = 47` 的含义是：**窗口右端 = 当日 15:00 收盘**，往左取满
`lookback_bars = 96` 根 5 分钟 bar（= 2 个交易日）。

```
        前一交易日 (48 根)              当日 (48 根)
   ├────────────────────────┼────────────────────────┤
   slot 0 ............... 47  slot 0 ............. 47 ← 锚点 t
   └──────────── 回看窗口 L = 96 根 ────────────────┘
```

切片就是 `X[:, t-L+1 : t+1]`，对全部股票同时取，形状 `(N, 96, 19)`。

In [ ]:
# [离线可跑] 窗口切片的下标关系
t = int(sel_t[-1])          # 最后一天的锚点
L = lookback
print(f'锚点 t_idx = {t} -> day_idx = {t // bars_per_day}, slot_idx = {t % bars_per_day}')
print(f'窗口切片   = X[:, {t - L + 1} : {t + 1}]  (长度 {L} 根 bar = {L // bars_per_day} 个交易日)')
print(f'窗口起点   : day_idx={(t - L + 1) // bars_per_day}, slot={(t - L + 1) % bars_per_day}')
print(f'窗口终点   : day_idx={t // bars_per_day}, slot={t % bars_per_day}  <- 当日 15:00 收盘')

## 第 4 步：把 `model_cfg['N']` 覆盖成线上股票数

权重是在本地股票轴（2074 只）上训的，线上股票轴来自 `instruments` 表，数量通常不同。
不覆盖 `N` 会直接 reshape 报错。

能这么覆盖的依据（`demo_train.ipynb` 里验证过）：`N` 只是 `forward` 末尾
`h.view(-1, N)` 的 reshape 参数，backbone 不接收 `N`，每只股票的窗口独立前向、
无跨股票混合 —— 所以 `N` 不影响参数量和 `state_dict` 形状，同一份权重换个 `N` 仍能
`strict=True` 加载。

位置→股票的对应关系用的是面板自己返回的 `arrays['keys']`，覆盖 `N` 不引入错位。

关于"预测是否完全一样"，下面的 cell 会实测，这里先说结论：**同一个 `N` 重复跑是
逐位相同的**（模型本身确定）；把同一批股票放进更大的 `N`（尾部补零）时，预测在
float32 精度内相同，但不一定逐位相同 —— batch 形状变了会让矩阵乘用不同的分块/
归约顺序，末位有 1 ULP 级别的差异（实测 `max|Δ| ≈ 1.5e-8`，量级远小于预测值本身
的 1e-2，对排序无实质影响）。这属于浮点结合律，不是错位。

In [ ]:
# [离线可跑] 换 N 加载同一份权重, 实测预测差异有多大
n_panel = 37                    # 假设线上面板只有 37 只 (演示用小数字)
ck_a = A.load_model(A.MODEL_PATH, map_location='cpu')
ck_b = A.load_model(A.MODEL_PATH, map_location='cpu')
ck_a['model_cfg'] = {**ck_a['model_cfg'], 'N': n_panel}
ck_b['model_cfg'] = {**ck_b['model_cfg'], 'N': n_panel + 5}   # 换一个 N

m_a = A.build_model_from_checkpoint(ck_a, map_location='cpu').eval()
m_b = A.build_model_from_checkpoint(ck_b, map_location='cpu').eval()
print(f"N={n_panel} 与 N={n_panel + 5}: 参数量 "
      f"{sum(p.numel() for p in m_a.parameters()):,} vs "
      f"{sum(p.numel() for p in m_b.parameters()):,}  <- 相同, strict=True 都能加载")

g = torch.Generator().manual_seed(7)
X = torch.randn(1, n_panel, int(ckpt['seq_len']), A.N_FEAT, generator=g)
mask = torch.ones(1, n_panel, dtype=torch.bool)
with torch.no_grad():
    pa = m_a(X, pad_mask=mask).numpy().ravel()
    pa2 = m_a(X, pad_mask=mask).numpy().ravel()          # 同 N 重复一次
    # 同一批股票放进更大的 N (尾部补零), 取前 n_panel 个位置
    Xb = torch.cat([X, torch.zeros(1, 5, X.shape[2], X.shape[3])], dim=1)
    mb = torch.cat([mask, torch.zeros(1, 5, dtype=torch.bool)], dim=1)
    pb = m_b(Xb, pad_mask=mb).numpy().ravel()[:n_panel]

d = np.abs(pa - pb)
print(f'\n同一个 N 重复两次      : 逐位相同 = {np.array_equal(pa, pa2)}  <- 模型本身确定')
print(f'换 N (尾部补零) 后     : 逐位相同 = {np.array_equal(pa, pb)}, '
      f'不同的位置 {int((pa != pb).sum())}/{n_panel}')
print(f'  max|Δ| = {d.max():.3e}   (预测值本身量级 ~{np.abs(pa).mean():.2e})')
print(f'  allclose(rtol=1e-6) = {np.allclose(pa, pb, rtol=1e-6, atol=1e-8)}')
print('\n差异来自 batch 形状变化导致的矩阵乘分块/归约顺序不同 (浮点结合律),')
print('是 float32 末位 1 ULP 级别的舍入, 不是错位; 对截面排序无实质影响。')
print('\n样例预测:', np.array2string(pa[:4], precision=8))

## 第 5 步：还原量纲 + 对齐中证 1000

模型输出的是**标准化后**的预测值（训练时 label 做过 z-score），要用存盘的
`y_mean`/`y_std` 还原成真实收益率量纲：

```python
pred = pred * y_std + y_mean
```

然后只保留 `X_mask` 为 True 的股票（回看窗口内无残留缺失）—— 注意**未来 label
不参与筛选**（线上没有未来数据，这与回测用的 `build_prediction_table` 不同，
那里因为要算指标才同时要求 `y_mask`）。

最后与 `instruments` 表按 `(date, instrument)` 做 inner merge 对齐中证 1000 成分股，
去掉 inf/NaN、去重，输出 `['date','instrument','score']`。这一段与
`example_predict.py` 是同一段逻辑。

In [ ]:
# [离线可跑] 还原量纲这一步的算术
y_mean, y_std = float(ckpt['y_mean']), float(ckpt['y_std'])
raw = np.array([-2.0, -0.5, 0.0, 0.5, 2.0], np.float32)      # 标准化空间的预测
real = raw * y_std + y_mean
print('标准化输出 ->  真实收益率量纲')
for r, v in zip(raw, real):
    print(f'  {r:+.2f}  ->  {v:+.6f}  ({v * 100:+.3f}%)')
print(f'\n(y_mean={y_mean:.6f}, y_std={y_std:.6f}, 均来自 checkpoint, 非测试区间重估)')

## 第 6 步：真正打分

上面拆的每一步，`main`（= `predict_scores`）内部都会做一遍。

> **下面这个 cell 需要平台数据**（`dai.query` 拉面板 + `instruments` 表）。
> 默认 `RUN = False`，改成 `True` 再执行。

In [ ]:
# [需要平台数据] 真正打分。默认不跑, 改 RUN = True 再执行。
RUN = True

start_date, end_date = '2024-01-01 00:00:00', '2024-12-31 23:59:59'

if RUN:
    score_data = A.main(A.DATASOURCES, start_date, end_date)
    print(score_data.head())
    print()
    print('列名   :', list(score_data.columns), '(平台要求的顺序)')
    print('行数   :', len(score_data))
    print('交易日 :', score_data['date'].nunique())
    print('股票数 :', score_data['instrument'].nunique())
    # 每天应当恰好一个截面, 每只股票每天一个分数
    per_day = score_data.groupby('date').size()
    print(f'每日股票数: 最少 {per_day.min()}, 最多 {per_day.max()}, 中位 {int(per_day.median())}')
else:
    print('RUN = False, 未打分。')
    print('提交用的正式打分请直接跑 competition_predict.ipynb '
          '(或 competition_all_in_one.py predict)')

## 第 7 步：评估

拿到 `score_data` 后用平台的评估系统看绩效：分数经风格剔除后等价于每日单因子，
`show=True` 会画 IC / 分组 / 压力期等图。

> **需要平台环境**（`bigmodule`）。同样默认不跑。

In [ ]:
# [需要平台数据] 评估。默认不跑, 改 RUN_EVAL = True 并确保上一步已产出 score_data。
RUN_EVAL = True

if RUN_EVAL:
    from bigmodule import M
    result = M.bigalpha_eval._latest(factor_data=score_data, show=True)
else:
    print('RUN_EVAL = False, 未评估。')

## 小结

1. **只推理不训练** —— 权重来自随文件上传的 JSON
2. **四道断言** —— 配置与权重错配时直接报错，不静默算出错误分数
3. **每天固定 slot 47** —— 按 `slot_idx` 精确过滤，不用"每天取最大 `t_idx`"
   （后者在换配置时会静默打到 14:50）
4. **标准化统计复用 checkpoint 的** —— 不用测试区间重估，杜绝泄漏与漂移
5. **`N` 按面板股票数覆盖** —— `N` 不影响参数量，不引入错位
6. **输出 `['date','instrument','score']`** —— 对齐中证 1000，与 example 同一段逻辑

配套 `demo_train.ipynb` 讲训练侧。正式提交仍然跑 `competition_train.ipynb` /
`competition_predict.ipynb`（本 notebook 只是把同一套代码拆开讲）。